### **Analisando Artigos**

In [1]:
# Importando Blibiotecas
import os
import json
import pandas as pd
import requests
import time
import json
from pathlib import Path

In [ ]:
# Definindo a query
query = (
    "([videofluoroscopy] OR [videofluoroscopic swallowing study] OR "
    "[videofluoroscopic swallowing studies] OR [VFSS] OR "
    "[modified barium swallow] OR [MBS] OR [barium swallow study]) "
    "AND "
    "([artificial intelligence] OR [machine learning] OR [deep learning] OR "
    "[neural network] OR [convolutional neural network] OR [CNN] OR "
    "[computer vision] OR [image recognition] OR [video analysis] OR "
    "[object detection] OR [pose estimation] OR [optical flow] OR "
    "[segmentation] OR [tracking] OR [classification] OR [prediction]) "
    "AND "
    "([swallowing] OR [deglutition] OR [dysphagia] OR "
    "[aspiration] OR [penetration] OR [pharyngeal] OR [laryngeal] OR "
    "[hyoid] OR [epiglottis] OR [bolus] OR "
    "[upper esophageal sphincter] OR [UES] OR "
    "[swallowing kinematics] OR [swallowing timing] OR [swallowing events] OR " 
    "[cervical] OR [vertebrae] OR [mandibule] OR [residue] OR "
    "[deglutition disorders] OR [laryngeal closure])"
)

In [3]:
# Fazendo a busca

output_json = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\vfss_ai_papers.json"

ret = os.system(
    f'findpapers search "{output_json}" -q "{query}"'
)

if ret:
    print("ERRO no comando")
else:
    print("Seleção Concluída!")

Seleção Concluída!


In [4]:
# Aumentando as informações do Json encontrado.

def buscar_doi_pubmed(titulo: str) -> tuple[str | None, str | None]:
    """Busca DOI e URL no PubMed pela API Entrez usando o título do artigo."""
    try:
        # 1. Busca o PMID pelo título
        search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        r = requests.get(search_url, params={
            "db": "pubmed", "term": titulo, "retmode": "json", "retmax": 1
        }, timeout=10)
        ids = r.json().get("esearchresult", {}).get("idlist", [])
        if not ids:
            return None, None

        pmid = ids[0]

        # 2. Busca os detalhes pelo PMID
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
        r2 = requests.get(fetch_url, params={
            "db": "pubmed", "id": pmid, "retmode": "json"
        }, timeout=10)
        result = r2.json().get("result", {}).get(pmid, {})

        doi = None
        for id_obj in result.get("articleids", []):
            if id_obj.get("idtype") == "doi":
                doi = id_obj.get("value")
                break

        url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else None
        return doi, url

    except Exception as e:
        return None, None

def enriquecer_json(search_json_path: str, delay: float = 0.4):
    """
    Percorre todos os artigos do JSON de busca e preenche doi + urls
    nos que estiverem vazios, consultando o PubMed.
    Salva o JSON enriquecido no mesmo arquivo.
    """
    with open(search_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    atualizados = 0
    sem_resultado = []

    for i, p in enumerate(data["papers"]):
        tem_doi  = bool(p.get("doi"))
        tem_urls = bool(p.get("urls"))

        if tem_doi and tem_urls:
            continue  # já está completo

        titulo = p.get("title", "")
        print(f"[{i+1}/{len(data['papers'])}] Buscando: {titulo[:70]}...")

        doi, url = buscar_doi_pubmed(titulo)

        if doi:
            p["doi"] = doi
            atualizados += 1
            print(f"  ✅ DOI encontrado: {doi}")
        else:
            sem_resultado.append(titulo)
            print(f"  ⚠️  Sem DOI")

        if url:
            urls = list(p.get("urls") or [])
            if url not in urls:
                urls.append(url)
            if doi:
                doi_url = f"http://doi.org/{doi}"
                if doi_url not in urls:
                    urls.append(doi_url)
            p["urls"] = urls

        time.sleep(delay)  # respeita o rate limit da API do PubMed (max 3 req/s)

    with open(search_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Enriquecimento concluído!")
    print(f"   Atualizados : {atualizados}")
    print(f"   Sem resultado: {len(sem_resultado)}")
    if sem_resultado:
        print("\nArtigos sem DOI encontrado:")
        for t in sem_resultado:
            print(f"  - {t}")

# ── Executa ──────────────────────────────────────────────────
enriquecer_json(output_json)

[1/228] Buscando: Expert Consensus Statement on Acoustic Metrics for Swallowing Dysfunct...
  ✅ DOI encontrado: 10.1016/j.jvoice.2026.05.036
[2/228] Buscando: Deep learning for early detection of Zenker's diverticulum based on sw...
  ✅ DOI encontrado: 10.1007/s11548-026-03727-8
[3/228] Buscando: Deep Learning-Based Acoustic Screening for Penetration-Aspiration Even...
  ✅ DOI encontrado: 10.1007/s00455-026-10956-1
[4/228] Buscando: Post-swallowing voice-based aspiration screening in dysphagia using a ...
  ✅ DOI encontrado: 10.1038/s41598-026-53618-w
[5/228] Buscando: SPARNet: A Framework for Airway Invasion Tracking from Fluoroscopic Vi...
  ✅ DOI encontrado: 10.1109/JBHI.2026.3695144
[6/228] Buscando: Swallowing impairment and aspiration risk in clinically stabilized pat...
  ✅ DOI encontrado: 10.3389/fmed.2026.1804250
[7/228] Buscando: YOLO11-based detection of manometry sensors in video-fluoroscopy imagi...
  ✅ DOI encontrado: 10.3389/fradi.2026.1767875
[8/228] Buscando: Objective

In [5]:
# Examinando artigos encontrados

with open(output_json, "r", encoding="utf-8") as f:
    papers = json.load(f)

df = pd.DataFrame(papers["papers"])

print(f"{len(df)} artigos encontrados")
df.head()

228 artigos encontrados


,abstract,authors,categories,citations,comments,databases,doi,keywords,number_of_pages,pages,publication,publication_date,selected,title,urls
0,OBJECTIVE OBJECTIVE Swallowing dysfunction pos...,"[Adrián Castillo-Allendes, Sara W Albert, Jame...",None,None,None,[PubMed],10.1016/j.jvoice.2026.05.036,"[N Dysphagia, N Expert consensus, N Swallowing...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-06-16,None,Expert Consensus Statement on Acoustic Metrics...,"[https://pubmed.ncbi.nlm.nih.gov/42303507/, ht..."
1,PURPOSE OBJECTIVE Patients with Zenker's diver...,"[Daniel Ostler-Mildner, Alissa Jell, Matthias ...",None,None,None,[PubMed],10.1007/s11548-026-03727-8,"[N Dysphagia, N Biomedical acoustics, N Comput...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-06-03,None,Deep learning for early detection of Zenker's ...,"[https://pubmed.ncbi.nlm.nih.gov/42234064/, ht..."
2,To evaluate the feasibility of a smartphone-ba...,"[Yong Jae Na, Jun Hyeok Lee, Eunyoung Choi, Jo...",None,None,None,[PubMed],10.1007/s00455-026-10956-1,"[N Artificial intelligence, N Machine learning...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-06-02,None,Deep Learning-Based Acoustic Screening for Pen...,"[https://pubmed.ncbi.nlm.nih.gov/42228093/, ht..."
3,Dysphagia presents a serious risk of aspiratio...,"[Jung-Min Kim, Min-Seop Kim, Sun-Young Choi, H...",None,None,None,[PubMed],10.1038/s41598-026-53618-w,"[N Voice analysis, N Real-time monitoring, N D...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-05-21,None,Post-swallowing voice-based aspiration screeni...,"[https://pubmed.ncbi.nlm.nih.gov/42168445/, ht..."
4,The videofluoroscopic swallowing study (VFSS) ...,"[Sanjeevi G, Uma Gopalakrishnan, Rahul Krishna...",None,None,None,[PubMed],10.1109/JBHI.2026.3695144,[],NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-05-19,None,SPARNet: A Framework for Airway Invasion Track...,"[https://pubmed.ncbi.nlm.nih.gov/42154713/, ht..."


In [6]:
# Verificando algumas informações

df.columns.tolist()

cols = [c for c in df.columns if any(
    k in c.lower()
    for k in ["title", "author", "year", "doi", "abstract"]
)]

df[cols].head()

,abstract,authors,doi,title
0,OBJECTIVE OBJECTIVE Swallowing dysfunction pos...,"[Adrián Castillo-Allendes, Sara W Albert, Jame...",10.1016/j.jvoice.2026.05.036,Expert Consensus Statement on Acoustic Metrics...
1,PURPOSE OBJECTIVE Patients with Zenker's diver...,"[Daniel Ostler-Mildner, Alissa Jell, Matthias ...",10.1007/s11548-026-03727-8,Deep learning for early detection of Zenker's ...
2,To evaluate the feasibility of a smartphone-ba...,"[Yong Jae Na, Jun Hyeok Lee, Eunyoung Choi, Jo...",10.1007/s00455-026-10956-1,Deep Learning-Based Acoustic Screening for Pen...
3,Dysphagia presents a serious risk of aspiratio...,"[Jung-Min Kim, Min-Seop Kim, Sun-Young Choi, H...",10.1038/s41598-026-53618-w,Post-swallowing voice-based aspiration screeni...
4,The videofluoroscopic swallowing study (VFSS) ...,"[Sanjeevi G, Uma Gopalakrishnan, Rahul Krishna...",10.1109/JBHI.2026.3695144,SPARNet: A Framework for Airway Invasion Track...


In [7]:
# Salvando informações em um excel

df.to_excel(
    r"..\data\artigos\vfss_ai_papers.xlsx",
    index=False
)

print("Arquivo salvo!")

Arquivo salvo!
